# MorganTrace — Modelado y Entrenamiento

**Modelos:** XGBoost + LightGBM con seguimiento MLflow  
**Métricas objetivo:** ROC-AUC ≥ 0.90, F1 ≥ 0.80  
**Autor:** Jean Pierre Azabache  

## Flujo de trabajo
1. Cargar datos procesados del notebook 02
2. Entrenar XGBoost con MLflow
3. Entrenar LightGBM con MLflow
4. Comparar métricas y seleccionar mejor modelo
5. Guardar modelo ganador en `src/models/model.pkl`


In [ ]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
import joblib
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score,
    recall_score, roc_curve
)
import matplotlib.pyplot as plt
import os

os.makedirs('src/models', exist_ok=True)
os.makedirs('notebooks/figuras', exist_ok=True)
print('✅ Librerías cargadas')

In [ ]:
# ── Cargar datos procesados (del notebook 02) ──────────────────────────────────
X_train = np.load('data/processed/X_train.npy')
y_train = np.load('data/processed/y_train.npy')
X_test  = np.load('data/processed/X_test.npy')
y_test  = np.load('data/processed/y_test.npy')

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Fraude en test: {y_test.mean()*100:.2f}%')

In [ ]:
# ── Configurar MLflow ──────────────────────────────────────────────────────────
mlflow.set_tracking_uri('mlflow/')
mlflow.set_experiment('MorganTrace_Fraud_Detection')
print('✅ MLflow configurado en mlflow/')

In [ ]:
# ── Entrenar XGBoost ───────────────────────────────────────────────────────────
with mlflow.start_run(run_name='XGBoost_v1'):
    params_xgb = {
        'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'scale_pos_weight': 1,  # Datos ya balanceados con SMOTE
        'eval_metric': 'auc', 'random_state': 42, 'n_jobs': -1,
    }
    mlflow.log_params(params_xgb)

    modelo_xgb = XGBClassifier(**params_xgb)
    modelo_xgb.fit(X_train, y_train, verbose=False)

    y_pred_xgb   = modelo_xgb.predict(X_test)
    y_proba_xgb  = modelo_xgb.predict_proba(X_test)[:, 1]

    metricas_xgb = {
        'roc_auc'   : roc_auc_score(y_test, y_proba_xgb),
        'f1'        : f1_score(y_test, y_pred_xgb),
        'precision' : precision_score(y_test, y_pred_xgb),
        'recall'    : recall_score(y_test, y_pred_xgb),
    }
    mlflow.log_metrics(metricas_xgb)
    mlflow.sklearn.log_model(modelo_xgb, 'xgboost_model')

    print('─' * 50)
    print('XGBoost — Métricas en conjunto de prueba:')
    for k, v in metricas_xgb.items():
        print(f'  {k:<12}: {v:.4f}')

In [ ]:
# ── Entrenar LightGBM ──────────────────────────────────────────────────────────
with mlflow.start_run(run_name='LightGBM_v1'):
    params_lgbm = {
        'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05,
        'subsample': 0.8, 'colsample_bytree': 0.8,
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    mlflow.log_params(params_lgbm)

    modelo_lgbm = LGBMClassifier(**params_lgbm)
    modelo_lgbm.fit(X_train, y_train)

    y_pred_lgbm  = modelo_lgbm.predict(X_test)
    y_proba_lgbm = modelo_lgbm.predict_proba(X_test)[:, 1]

    metricas_lgbm = {
        'roc_auc'   : roc_auc_score(y_test, y_proba_lgbm),
        'f1'        : f1_score(y_test, y_pred_lgbm),
        'precision' : precision_score(y_test, y_pred_lgbm),
        'recall'    : recall_score(y_test, y_pred_lgbm),
    }
    mlflow.log_metrics(metricas_lgbm)
    mlflow.sklearn.log_model(modelo_lgbm, 'lightgbm_model')

    print('─' * 50)
    print('LightGBM — Métricas en conjunto de prueba:')
    for k, v in metricas_lgbm.items():
        print(f'  {k:<12}: {v:.4f}')

In [ ]:
# ── Seleccionar y guardar el mejor modelo ──────────────────────────────────────
if metricas_xgb['roc_auc'] >= metricas_lgbm['roc_auc']:
    mejor_modelo, mejor_nombre, mejores_metricas = modelo_xgb, 'XGBoost', metricas_xgb
else:
    mejor_modelo, mejor_nombre, mejores_metricas = modelo_lgbm, 'LightGBM', metricas_lgbm

# ⚠️ Verificar umbral mínimo antes de proceder al despliegue
assert mejores_metricas['roc_auc'] >= 0.85, \
    f"ROC-AUC ({mejores_metricas['roc_auc']:.4f}) por debajo del umbral mínimo (0.85)"

joblib.dump(mejor_modelo, 'src/models/model.pkl')

print('─' * 50)
print(f'✅ Mejor modelo: {mejor_nombre}')
print(f'   Guardado en: src/models/model.pkl')
print(f'\nMétricas finales:')
for k, v in mejores_metricas.items():
    print(f'  {k:<12}: {v:.4f}')

In [ ]:
# ── Curva ROC comparativa ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))

for proba, nombre, metricas, color in [
    (y_proba_xgb,  'XGBoost',  metricas_xgb,  '#e74c3c'),
    (y_proba_lgbm, 'LightGBM', metricas_lgbm, '#3498db'),
]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{nombre} (AUC = {metricas["roc_auc"]:.4f})')

ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Aleatorio (AUC = 0.50)')
ax.set_xlabel('Tasa de Falsos Positivos', fontsize=12)
ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=12)
ax.set_title('Curva ROC — MorganTrace', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('notebooks/figuras/03_curva_roc.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Modelado completado. Proceder con: bash k3d/setup.sh')